In [ ]:
import json
import os
from typing import List

import fitz
from fitz.utils import get_pixmap
from IPython.display import HTML, Markdown, display
from PIL import Image

from uniparser_tools.api.clients import UniParserClient
from uniparser_tools.common.constant import FormatFlag, LayoutType, ParseMode, ParseModeTextual
from uniparser_tools.common.dataclass import BBox, Direction, GroupedResult, LayoutItem, SemanticItem
from uniparser_tools.utils.convert import dict2obj
from uniparser_tools.utils.log import get_root_logger
from uniparser_tools.utils.processor import tree_repr


###### 以下为示例代码，请自行修改

# ==============================================================================================
host = "https://uniparser.dp.tech/"  # 官网

# 替换为你的认证api key
api_key = os.getenv('UNIPARSER_API_KEY')

# 初始化客户端
parser = UniParserClient(host=host, api_key=api_key)

token = "<TASK_TOKEN>"
input_file = "./tasks/CN1275981A.pdf"
save_dir = "./outputs/molecule_extracrtion"
os.makedirs(save_dir, exist_ok=True)
os.makedirs(f"{save_dir}/{token}", exist_ok=True)

In [ ]:
trigger_result = parser.trigger_file(
    file_path=input_file,
    token=token,
    textual=ParseModeTextual.DigitalExported,
    table=ParseMode.OCRFast,
    molecule=ParseMode.OCRFast,
    chart=ParseMode.DumpBase64,
    figure=ParseMode.DumpBase64,
    expression=ParseMode.DumpBase64,
    equation=ParseMode.OCRFast,
)
if trigger_result["status"] != "success":
    print(json.dumps(trigger_result, indent=4))
    raise Exception("trigger file failed")
print(f"trigger success, token: {trigger_result['token']}")

In [ ]:
result = parser.get_result(token, pages_tree=True)
if result["status"] != "success":
    print(json.dumps(result, indent=4))
    raise Exception("get result failed")
json.dump(result["pages_tree"], open(f"{save_dir}/{token}.json", "w"), indent=4)

In [ ]:
pages_tree = dict2obj(result["pages_tree"]) 

In [ ]:
pages_tree[2]

In [ ]:
print(tree_repr(GroupedResult.clone(pages_tree[2][0], type=LayoutType.Page, items=pages_tree[2])))

In [ ]:
pages_tree[2][0].items[0]

In [ ]:
mol_group = pages_tree[2][0].items[0]

In [ ]:
print(tree_repr(mol_group))

In [ ]:
mol_group.items[0]

In [ ]:
display(Markdown(mol_group.format_as(FormatFlag.Markdown)))

In [ ]:
display(HTML(f'<img src="data:image/png;base64,{mol_group.items[0].source}" />'))

In [ ]:
doc = fitz.Document(input_file)
# dpi = 300

# group 
# page = doc[group.page]
# max_dpi = min(dpi, max(1, int(4096 * 72 / max(page.rect.width, page.rect.height))))  # max 4096 pixels
# group_clip: BBox = group.bbox * [page.rect.width, page.rect.height] + tuple(page.rect.top_left)
# pix = get_pixmap(page, clip=fitz.Rect(*group_clip.xyxy), dpi=max_dpi)
# group_image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
# if save_dir:
#     pix.save(save_dir / f"{group_name}.group.png")
# group_size = [pix.width, pix.height]


def crop_item(
    page: fitz.Page, item: LayoutItem, dpi: int = 144, max_size: int = None, rotated: bool = True
) -> Image.Image:
    bbox: BBox = item.bbox * [page.rect.width, page.rect.height] + tuple(page.rect.top_left)
    clip = fitz.Rect(*bbox.xyxy)
    if max_size is not None:
        max_dpi = min(dpi, max(1, int(max_size * 72 / max(clip.width, clip.height))))  # max pixels
    else:
        max_dpi = dpi
    pix = get_pixmap(page=page, alpha=False, dpi=max_dpi, clip=clip)
    cropped = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    if min(cropped.size) == 0:
        get_root_logger().warning(
            f"{item.token} Crop config: {bbox=}, {clip=}, {page.rect=}, pix.shape={pix.width, pix.height}, {max_dpi=}"
        )
        
    if min(cropped.size) == 0:
        get_root_logger().warning(f"{item.token} Crop error: size={cropped.size} {item=}, {dpi=}, {max_size=}")
        cropped = Image.new("RGB", [10, 10], (255, 255, 255))
    if rotated and item.direction in [Direction.Rotate_90, Direction.Rotate_180, Direction.Rotate_270]:
        get_root_logger().debug(f"{item.token} Crop rotate: {item.direction=}")
        if item.direction == Direction.Rotate_90:
            cropped = cropped.transpose(Image.Transpose.ROTATE_90)
        elif item.direction == Direction.Rotate_180:
            cropped = cropped.transpose(Image.Transpose.ROTATE_180)
        elif item.direction == Direction.Rotate_270:
            cropped = cropped.transpose(Image.Transpose.ROTATE_270)
    return cropped


In [ ]:
def recursive_find_groups(
    item: SemanticItem,
    required_types: List[LayoutType] = [
        LayoutType.MoleculeGroup,
    ],
) -> List[SemanticItem]:
    for t in required_types:
        assert 'group' in t.value, t
    if item.type in required_types:
        return [item]
    elif isinstance(item, GroupedResult):
        items = [itt for it in item.items for itt in recursive_find_groups(it, required_types)]
        return items
    else:
        return []

In [ ]:
all_mol_groups: List[GroupedResult] = []
for page_idx, page in enumerate(pages_tree):
    # print(f"page {page_idx}")
    for item in page:
        # print(item)
        all_mol_groups.extend(recursive_find_groups(item, required_types=[LayoutType.MoleculeGroup]))
print(len(all_mol_groups))

In [ ]:
all_pairs = []
for mol_group_ in all_mol_groups:
    # print(tree_repr(mol_group_))
    # display(HTML(f'<span>分子</span> <img src="data:image/png;base64,{mol_group_.items[0].source}" />'))
    mol_group_image = crop_item(doc[mol_group_.page], mol_group_, dpi=200, max_size=512)
    mol_group_image.save(f"{save_dir}/{token}/{mol_group_.page}_{mol_group_.order}.png")
    mol_group_desc = mol_group_.format_as(FormatFlag.Markdown)
    open(f"{save_dir}/{token}/{mol_group_.page}_{mol_group_.order}.txt", "w").write(mol_group_desc)
    
    for mol in mol_group_.items:
        if getattr(mol, 'type', '') in ('molecule', LayoutType.Molecule):
            smiles = getattr(mol, 'smi', getattr(mol, 'plain', ''))
            caption = getattr(mol, 'caption', getattr(mol, 'plain', ''))
            markush = getattr(mol, 'markush', False)
            # 有时候 plain 属性包含 <sep> 后面的占位符内容，因此如果有明确的 smi 优先用 smi
            # 如果没有 smi 只能用 plain 时，尝试将 <sep> 后面的去掉以得到纯净的 smiles
            if smiles and '<sep>' in smiles:
                smiles = smiles.split('<sep>')[0]
                
            mol_id = ''
            if hasattr(mol, 'items'):
                for sub in mol.items:
                    t = getattr(sub, 'type', '')
                    if t == 'SMILES':
                        smiles = getattr(sub, 'smi', getattr(sub, 'plain', smiles))
                    elif t in ('moleculeid', LayoutType.MoleculeID):
                        mol_id = getattr(sub, 'text', getattr(sub, 'plain', mol_id))
            
            if not mol_id:
                for peer in mol_group_.items:
                    if peer != mol and getattr(peer, 'type', '') in ('moleculeid', LayoutType.MoleculeID):
                        mol_id = getattr(peer, 'text', getattr(peer, 'plain', mol_id))
                        break
                        
            # 针对没有明显 moleculeid 但却提取出了其他杂乱 caption 的情况，清除它
            if mol_id and '<sep>' in mol_id:
                mol_id = ''
                
            if smiles:
                all_pairs.append({
                    'index': len(all_pairs),
                    'molecular_info': [mol_id.strip()] if mol_id else [],
                    'SMILES': smiles.strip() if smiles else smiles,
                    'E-SMILES': caption.strip() if caption else caption,
                    'is_markush': bool(markush),
                })
    
    display(mol_group_image)
    # display(Markdown(mol_group_desc))
    print("==" * 50)

print("\n提取到的所有 id-smiles 分子对信息：")
print(json.dumps(all_pairs, indent=4, ensure_ascii=False))